In [0]:
# ============================================================
# Notebook 2: ETL Pipeline
# ATP Tennis Dataset - Cleaning & Transformation
# ============================================================

storage_account = "tennisdatalake60105845"
storage_key = "LjZoIlf5yGSIHxM/9GIXHx5jTq8VNvMOqWjuLZ54krBy3gn+BN7pg8q5/4UM8dgtAEwIMaTNyIYl+AStqzCGpQ=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# Load raw data from processed zone
df = spark.read.format("parquet") \
    .load(f"abfss://processed@{storage_account}.dfs.core.windows.net/tennis/raw_ingested/")

print(f"Loaded {df.count()} records")
df.printSchema()

Loaded 67460 records
root
 |-- Tournament: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- Series: string (nullable = true)
 |-- Court: string (nullable = true)
 |-- Surface: string (nullable = true)
 |-- Round: string (nullable = true)
 |-- Best of: integer (nullable = true)
 |-- Player_1: string (nullable = true)
 |-- Player_2: string (nullable = true)
 |-- Winner: string (nullable = true)
 |-- Rank_1: integer (nullable = true)
 |-- Rank_2: integer (nullable = true)
 |-- Pts_1: integer (nullable = true)
 |-- Pts_2: integer (nullable = true)
 |-- Odd_1: double (nullable = true)
 |-- Odd_2: string (nullable = true)
 |-- Score: string (nullable = true)



In [0]:
# ============================================================
# ETL Step 1: Handle Missing Values & Data Quality
# ============================================================
from pyspark.sql.functions import col, when, count, isnan

# Check missing values in each column
print("=== Missing Values Report ===")
for column in df.columns:
    missing = df.filter(col(column).isNull()).count()
    print(f"{column}: {missing} missing")

=== Missing Values Report ===
Tournament: 0 missing
Date: 0 missing
Series: 0 missing
Court: 0 missing
Surface: 0 missing
Round: 0 missing
Best of: 0 missing
Player_1: 0 missing
Player_2: 0 missing
Winner: 0 missing
Rank_1: 0 missing
Rank_2: 0 missing
Pts_1: 0 missing
Pts_2: 0 missing
Odd_1: 0 missing
Odd_2: 0 missing
Score: 0 missing


In [0]:
# ============================================================
# ETL Step 2: Clean & Transform Data
# ============================================================
from pyspark.sql.functions import col, year, month, when, trim, expr

# 1. Fix Odd_2 column using try_cast to handle '-' values
df_clean = df.withColumn("Odd_2", expr("try_cast(Odd_2 as double)"))

# 2. Add winner_is_player1 column (target variable for ML)
df_clean = df_clean.withColumn("winner_is_player1", 
    when(col("Winner") == col("Player_1"), 1).otherwise(0))

# 3. Extract year and month from Date
df_clean = df_clean.withColumn("year", year(col("Date")))
df_clean = df_clean.withColumn("month", month(col("Date")))

# 4. Remove duplicates
before = df_clean.count()
df_clean = df_clean.dropDuplicates()
after = df_clean.count()
print(f"Removed {before - after} duplicates")

# 5. Replace -1 ranks with None
df_clean = df_clean.withColumn("Rank_1", when(col("Rank_1") == -1, None).otherwise(col("Rank_1")))
df_clean = df_clean.withColumn("Rank_2", when(col("Rank_2") == -1, None).otherwise(col("Rank_2")))

print(f" Clean dataset: {df_clean.count()} records")
df_clean.show(3)

Removed 0 duplicates
 Clean dataset: 67460 records
+---------------+----------+----------+-------+-------+----------+-------+----------+-----------+--------+------+------+-----+-----+-----+-----+-----------+-----------------+----+-----+
|     Tournament|      Date|    Series|  Court|Surface|     Round|Best of|  Player_1|   Player_2|  Winner|Rank_1|Rank_2|Pts_1|Pts_2|Odd_1|Odd_2|      Score|winner_is_player1|year|month|
+---------------+----------+----------+-------+-------+----------+-------+----------+-----------+--------+------+------+-----+-----+-----+-----+-----------+-----------------+----+-----+
|   Chennai Open|2013-12-30|    ATP250|Outdoor|   Hard| 1st Round|      3|Smyczek T.|    Lu Y.H.| Lu Y.H.|    89|    65|  600|  730| 2.37| 1.53|    4-6 2-6|                0|2013|   12|
|Australian Open|2014-01-24|Grand Slam|Outdoor|   Hard|Semifinals|      5|  Nadal R.| Federer R.|Nadal R.|     1|     6|13130| 4355| 1.57| 2.37|7-6 6-3 6-3|                1|2014|    1|
|     Copa Claro|20

In [0]:
# ============================================================
# ETL Step 3: Save Cleaned Data to Processed Zone
# ============================================================

df_clean.write.format("parquet") \
    .mode("overwrite") \
    .partitionBy("year") \
    .save(f"abfss://processed@{storage_account}.dfs.core.windows.net/tennis/cleaned/")

print("Cleaned data saved to processed zone!")
print(f"Partitioned by year")
print(f"Total records: {df_clean.count()}")

Cleaned data saved to processed zone!
Partitioned by year
Total records: 67460


In [0]:
# ============================================================
# ETL Step 4: Data Cataloging & Governance
# ============================================================

# Fix column names - replace spaces with underscores
df_catalog = df_clean
for col_name in df_clean.columns:
    new_name = col_name.replace(" ", "_")
    df_catalog = df_catalog.withColumnRenamed(col_name, new_name)

print("Fixed column names:")
print(df_catalog.columns)

# Register the cleaned data as a table in Databricks catalog
df_catalog.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("amazon_dbx_60105845.default.atp_tennis_clean")

print(" Table registered in Databricks Catalog!")

# Document schema
print("\n=== DATA CATALOG ===")
print(f"Table name: atp_tennis_clean")
print(f"Schema: amazon_dbx_60105845.default")
print(f"Source: raw/atp_tennis.csv")
print(f"Records: {df_catalog.count()}")
print(f"Partitioned by: year")
print(f"\nColumn Definitions:")
for field in df_catalog.schema.fields:
    print(f"  - {field.name}: {field.dataType} | nullable={field.nullable}")

Fixed column names:
['Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best_of', 'Player_1', 'Player_2', 'Winner', 'Rank_1', 'Rank_2', 'Pts_1', 'Pts_2', 'Odd_1', 'Odd_2', 'Score', 'winner_is_player1', 'year', 'month']
 Table registered in Databricks Catalog!

=== DATA CATALOG ===
Table name: atp_tennis_clean
Schema: amazon_dbx_60105845.default
Source: raw/atp_tennis.csv
Records: 67460
Partitioned by: year

Column Definitions:
  - Tournament: StringType() | nullable=True
  - Date: DateType() | nullable=True
  - Series: StringType() | nullable=True
  - Court: StringType() | nullable=True
  - Surface: StringType() | nullable=True
  - Round: StringType() | nullable=True
  - Best_of: IntegerType() | nullable=True
  - Player_1: StringType() | nullable=True
  - Player_2: StringType() | nullable=True
  - Winner: StringType() | nullable=True
  - Rank_1: IntegerType() | nullable=True
  - Rank_2: IntegerType() | nullable=True
  - Pts_1: IntegerType() | nullable=True
  - Pts_2: Integer